# COVID-19 Disease Trend Analysis
### End-to-End Analytics with Python, SQL, and Power BI

**Author:** Anuj Kumar Kushwaha  
**Roll No:** 250321540033  
**Institution:** ABES Engineering College, Ghaziabad  
**Project Motto:** *"Analyse Today... Be Prepared Tomorrow..."*

---

## 1. Project Overview
The primary objective of this project is to analyze epidemiological data from the COVID-19 pandemic (2020–2023) sourced from **Our World in Data (WHO)**. We perform data cleaning, window moving averages, and cumulative aggregations in **Python**, execute analytical window queries in **SQL**, and model interactive multi-dimensional dashboards for **Power BI**.

## 2. Environment Setup & Data Ingestion

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set presentation style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

import os
# Load cleaned dataset
data_path = '../data/covid_data.csv' if os.path.exists('../data/covid_data.csv') else '../dataset/covid_data_cleaned.csv'
df = pd.read_csv(data_path)
df['date'] = pd.to_datetime(df['date'])

print(f"Dataset successfully loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

## 3. Data Cleaning & Quality Assurance

In [2]:
# Verify null counts and integrity
null_summary = df.isnull().sum()
print("Missing Value Count per Column:")
print(null_summary[null_summary > 0] if null_summary.sum() > 0 else "No missing values found! Dataset is 100% clean.")

## 4. Key Metrics & Global Aggregations

In [3]:
latest_date = df['date'].max()
df_latest = df[df['date'] == latest_date]

total_cases = df_latest['total_cases'].sum()
total_deaths = df_latest['total_deaths'].sum()
total_recovered = df_latest['total_recovered'].sum()

print("=" * 50)
print(f"Total Global Cases:     {total_cases / 1e6:.1f} Million ({total_cases:,})")
print(f"Total Global Deaths:    {total_deaths / 1e6:.2f} Million ({total_deaths:,})")
print(f"Total Global Recovered: {total_recovered / 1e6:.1f} Million ({total_recovered:,})")
print(f"Case Fatality Rate:     {(total_deaths / total_cases) * 100:.2f}%")
print(f"Recovery Rate:          {(total_recovered / total_cases) * 100:.2f}%")
print("=" * 50)

## 5. Epidemic Waves & 7-Day Moving Average Analysis

In [4]:
# Aggregate daily global sums
df_daily = df.groupby('date').agg({
    'new_cases': 'sum',
    'rolling_7day_cases': 'sum',
    'new_deaths': 'sum',
    'rolling_7day_deaths': 'sum'
}).reset_index()

plt.figure(figsize=(14, 6))
plt.plot(df_daily['date'], df_daily['new_cases'] / 1e6, label='Daily New Cases', color='#93c5fd', alpha=0.6)
plt.plot(df_daily['date'], df_daily['rolling_7day_cases'] / 1e6, label='7-Day Moving Average', color='#d97706', linewidth=2.5)
plt.title('Daily COVID-19 Cases vs 7-Day Moving Average (2020-2023)', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Cases (Millions)')
plt.legend()
plt.tight_layout()
plt.show()

## 6. Top Countries Comparison

In [5]:
top_5 = df_latest[df_latest['country'] != 'Other Countries'].sort_values('total_cases', ascending=False).head(5)

plt.figure(figsize=(10, 5))
bars = plt.barh(top_5['country'][::-1], (top_5['total_cases'] / 1e6)[::-1], color='#1d4ed8')
plt.title('Top 5 Countries by Cumulative Cases (Millions)', fontsize=14, fontweight='bold')
plt.xlabel('Cases in Millions')
for bar in bars:
    plt.text(bar.get_width() + 1, bar.get_y() + 0.3, f"{bar.get_width():.0f}M", fontweight='bold')
plt.xlim(0, 125)
plt.tight_layout()
plt.show()

## 7. Conclusions & Findings
- **Peak Waves:** Global cases surged drastically in early 2022 due to the Omicron variant, reaching single-day averages of over 3.4M cases.
- **Vaccination Decoupling:** In 2020-2021, spikes in cases caused corresponding severe spikes in fatalities. Post-2021 vaccination rollouts drastically reduced the Case Fatality Rate.
- **Regional Patterns:** USA and India reported the highest cumulative case counts globally.